In [1]:
import pyspark
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=C:/tools/hadoop/bin") \
    .getOrCreate()

In [3]:
# !curl -fL --create-dirs -o ../zones/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [4]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('../zones/taxi_zone_lookup.csv')

df_zones.createOrReplaceTempView('zones')

In [5]:
df_yellow = spark.read.parquet('../data/pq/yellow/2025/11/')

df_yellow.createOrReplaceTempView('yellow')

In [10]:
query = """
    select
        z.Zone,
        count(1) as trips
    from yellow y
    join zones z
        on y.PULocationID = z.LocationID
    group by
        z.Zone
    order by
        trips asc
"""

In [11]:
spark.sql(query).show()

+--------------------+-----+
|                Zone|trips|
+--------------------+-----+
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|New Dorp/Midland ...|   14|
|       West Brighton|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
